In [1]:
import toml
from pathlib import Path
config_path = Path("../../../../configuration/asim_configuration")
input_config = toml.load(config_path/ "input_configuration.toml")
summary_config = toml.load(config_path/ "summary_configuration.toml")

The individual mandatory tour frequency model **predicts the number of work and school tours taken by each person with a mandatory DAP**. The primary drivers of mandatory tour frequency are demographics, accessibility-based parameters such as drive time to work, and household automobile ownership.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [3]:
# read data
hh = util.get_hh_data(summary_config, uncloned=True)
person = util.get_person_data(summary_config, uncloned=True)
tour = util.get_tour_data(summary_config)

In [4]:
# count number of tours by tour category and type for each person
person_tour_cat = tour[tour['tour_cat_reassign']=="mandatory"].groupby(['person_id','tour_cat_reassign','source'])['tour_id'].size().\
    unstack('tour_cat_reassign').reset_index()
person_tour_type = tour[tour['tour_type'].isin(["work","school"])].groupby(['person_id','tour_type','source'], observed=True)['tour_id'].size().\
    unstack('tour_type').reset_index()
# cap work/school trips at 2
person_tour_type.loc[person_tour_type['work']>2,'work'] = 2
person_tour_type.loc[person_tour_type['school']>2,'school'] = 2

per_data = person.\
    merge(person_tour_cat, how='left', on=['person_id','source']).\
    merge(person_tour_type, how='left', on=['person_id','source']).\
    merge(hh[['household_id','income','num_drivers','auto_ownership','auto_ownership_4+','hhsize_4+','hh_weight','source']],
            how='left', on=['household_id','source']) # get auto ownership from hh data
per_data[['mandatory','work','school']] = per_data[['mandatory','work','school']].fillna(0)

tour_data = tour.merge(per_data, how='left', on=['person_id','household_id','source'])

In [5]:
# # get mandatory person data
# m_per_data = per_data[per_data['cdap_activity']=="M"].copy()

# # get mandatory tour data
# m_tour_data = tour_data[tour_data['tour_category']=="mandatory"].copy()
# work tours
w_tour_data = tour_data[tour_data['tour_type']=="work"].copy()
# school tours
s_tour_data = tour_data[tour_data['tour_type']=="school"].copy()

In [6]:
# total number of persons, mandatory persons, worker, student by source
df_counts = {"region": per_data.groupby('source')['person_weight'].sum().reset_index(), 
             "mandatory": per_data.loc[per_data['cdap_activity']=="M"].groupby('source')['person_weight'].sum().reset_index(), 
             "worker": per_data.loc[per_data['is_worker']==1].groupby('source')['person_weight'].sum().reset_index(), 
             "student": per_data.loc[per_data['is_student']==1].groupby('source')['person_weight'].sum().reset_index()}

In [7]:
# get tour rates
def calc_tour_rates(df_tour, group):
    """ Calculates tour rates for person groups.
    Parameters:
        df_tour: dataframe with tour data
        group: person group name in df_counts

    Returns:
        pandas dataframe for plotting
    """

    df_plot = df_tour.groupby(['source'],observed=True)[['tour_weight']].sum().reset_index(). \
        merge(df_counts[group], how='left', on='source')
    df_plot['tour rate'] = df_plot['tour_weight']/df_plot['person_weight']

    return(df_plot)

## work & school tour rates

- total student/worker count (using `is_worker` and `is_student`, can have intercept)

In [8]:
pd.DataFrame({'student': df_counts['student'].loc[df_counts['student']['source']=="model",'person_weight'],
              'worker': df_counts['worker'].loc[df_counts['worker']['source']=="model",'person_weight']})

,student,worker
0,1059394.0,1982128.0


In [9]:
# worker/student tour rates
df_plot = pd.concat([
    # worker
    calc_tour_rates(w_tour_data,'worker'),
    # student
    calc_tour_rates(s_tour_data,'student')
])
df_plot['worker/student'] = ["worker","worker","student","student"]
df_plot['tour rate'] = df_plot['tour_weight']/df_plot['person_weight']

fig = px.bar(df_plot, x="worker/student", y="tour rate", color="source",barmode="group",
             title="work/school tour rates by worker/student")
fig.update_layout(height=400, width=500)
fig.show()

## number of work tours by person type

In [10]:
worker_ptype = [1,2,3,6]
def plot_work_tour_ptype(person_data, title):
    util.plot_share_facetbar(person_data, 'person_weight', 'work', title, 
                        'ptype_label', facet_col_wrap=4,
                        height=350, orientation='v')

In [11]:
plot_work_tour_ptype(per_data[per_data['ptype'].isin(worker_ptype)],"number of work tours in a day by person type")

In [12]:
plot_work_tour_ptype(per_data[(per_data['ptype'].isin(worker_ptype)) & (per_data['sex']==2)],"Female: number of work tours in a day by person type")

In [13]:
plot_work_tour_ptype(per_data[(per_data['ptype'].isin(worker_ptype)) & (per_data['age']<=35)],"Under 35 years old: number of work tours in a day by person type")

In [14]:
plot_work_tour_ptype(per_data[(per_data['ptype'].isin(worker_ptype)) & (per_data['distance_to_work'] < 3)],"Walk to work: number of work tours in a day by person type")

In [15]:
plot_work_tour_ptype(per_data[(per_data['ptype'].isin(worker_ptype)) & (per_data['auto_ownership'] == 0)],"No cars in household: number of work tours in a day by person type")

In [16]:
plot_work_tour_ptype(per_data[(per_data['ptype'].isin(worker_ptype)) & (per_data['auto_ownership'] < per_data['num_drivers'])],"Fewer cars than drivers in household: number of work tours in a day by person type")

In [17]:
plot_work_tour_ptype(per_data[(per_data['ptype'].isin(worker_ptype)) & (per_data['income'] > 50000)],"Household income higher than $50k: number of work tours in a day by person type")

## number of school tours by person type

In [18]:
student_ptype = [3,6,7]
def plot_school_tour_ptype(person_data, title):
    util.plot_share_facetbar(person_data, 'person_weight', 'school', title, 
                        'ptype_label', facet_col_wrap=4,
                        height=350, orientation='v')

In [19]:
plot_school_tour_ptype(per_data[per_data['ptype'].isin(student_ptype)],"number of school tours in a day by person type")

In [20]:
plot_school_tour_ptype(per_data[per_data['ptype'].isin(student_ptype) & (per_data['sex']==2)],"Female: number of school tours in a day by person type")

In [21]:
plot_school_tour_ptype(per_data[per_data['ptype'].isin(student_ptype) & (per_data['age']<=35)],"Under 35 years old: number of school tours in a day by person type")

In [22]:
plot_school_tour_ptype(per_data[per_data['ptype'].isin(student_ptype) & (per_data['distance_to_work'] < 3)],"Walk to work: number of school tours in a day by person type")

In [23]:
plot_school_tour_ptype(per_data[per_data['ptype'].isin(student_ptype) & (per_data['auto_ownership'] == 0)],"No cars in household: number of school tours in a day by person type")

In [24]:
plot_school_tour_ptype(per_data[per_data['ptype'].isin(student_ptype) & (per_data['auto_ownership'] < per_data['num_drivers'])],"Fewer cars than drivers in household: number of school tours in a day by person type")

## number to work tours by distance to work

In [25]:
worker_ptype = [1,2,3,6]
def plot_work_tour_workdistance(person_data, title):
    util.plot_share_facetbar(person_data, 'person_weight', 'work', title, 
                        'distance_to_work_bin', facet_col_wrap=5,
                        height=350, orientation='v')

In [26]:
util.plot_share_facetbar(per_data[per_data['ptype']==1], 'person_weight', 'work', "Full-time worker: number of work tours in a day by distance to work", 
                    'distance_to_work_bin', facet_col_wrap=5,
                    height=350, orientation='v')

In [27]:
util.plot_share_facetbar(per_data[per_data['ptype']==2], 'person_weight', 'work', "Part-time worker: number of work tours in a day by distance to work", 
                    'distance_to_work_bin', facet_col_wrap=5,
                    height=350, orientation='v')

## number of school tours by distance to school

In [28]:
util.plot_share_facetbar(per_data[per_data['ptype'].isin(student_ptype)], 'person_weight', 'school', 
                         "All students: number of school tours in a day by distance to school", 
                         'distance_to_school_bin', facet_col_wrap=5,
                         height=350, orientation='v')

In [29]:
util.plot_share_facetbar(per_data[per_data['ptype']==3], 'person_weight', 'school', 
                         "University students: number of school tours in a day by distance to school", 
                         'distance_to_school_bin', facet_col_wrap=5,
                         height=350, orientation='v')